In [ ]:
import random, heapq, math, statistics
import matplotlib.pyplot as plt


# SYSTEM DATA
lam = 42                      # Αφίξεις ανα ώρα
mu  = 13.33                   # Μέσος χρόνος εξυπηρέτησης/ώρα/υπηρέτη
T15 = 15/60                   # Όριο SLA (15 minutes)
cost_per_min = 0.80           # Κόστος Καθυστέρησεις €/minute/προιόν

# USER INPUT
c = int(input("Δώσε αριθμό εργαζομένων μέχρι 6 (c): "))

# M/M/c ANALYTICS  
def mmc_metrics(lam, mu, c):
    Ro = lam / (c * mu)   #Ro=ρ

    if Ro >= 1:
        return {"stable": False, "Ro": Ro}

    r = lam / mu
    sum_terms = sum(r**n / math.factorial(n) for n in range(c)) # Σ = Σ (r^n / n!) για n = 0 έως c-1
    last = r**c / (math.factorial(c) * (1 - Ro))
    P0 = 1 / (sum_terms + last) # P0 = πιθανότητα να μην υπάρχει κανένα προιόν στο σύστημα

    Pwait = last * P0 # πιθανότητα ένα προιόν να περιμένει στην ουρά
    Lq = P0 * (r**c) * Ro / (math.factorial(c) * (1 - Ro)**2) # Lq = αναμενόμενος αριθμός προιόντων στην ουρά
    Wq = Lq / lam # Μέσος χρόνος αναμονής (νόμος Little)
    W  = Wq + 1/mu # Συνολικός χρόνος στο σύστημα

    return {
        "stable": True,
        "Ro": Ro,
        "Pwait": Pwait,
        "Lq": Lq,
        "Wq_h": Wq,
        "W_h": W
    }

# SLA: P(W < 15')
def sla_prob_mmc(lam, mu, c, t_hours):
    m = mmc_metrics(lam, mu, c)
    if not m.get("stable", False):
        return None

    Pwait = m["Pwait"]
    Pnowait = 1 - Pwait

    cdf_service = 1 - math.exp(-mu * t_hours) #Όταν δεν υπάρχει αναμονή, τότε:𝑊=𝑆 
                                              #Άρα:P(W < t|χωρίς αναμονή)= P(S < t)
    r1 = c*mu - lam
    r2 = mu
    cdf_sum = 1 - (r2*math.exp(-r1*t_hours) - r1*math.exp(-r2*t_hours)) / (r2 - r1)

    return Pnowait * cdf_service + Pwait * cdf_sum # συνολική πιθανότητα SLA

# ΚΟΣΤΟΣ ΚΑΘΥΣΤΕΡΗΣΗΣ
def delay_cost_per_hour(lam, Wq_hours):
    total_wait_min = lam * Wq_hours * 60 # συνολικά λεπτά αναμονής ανά ώρα
    return total_wait_min * cost_per_min # κόστος καθυστέρησης / ώρα

# ΠΡΟΣΟΜΟΙΩΣΗ (M/M/c) 
def simulate_mm_c(lam, mu, c, horizon_h=60, warmup_h=10, seed=0):
    random.seed(seed)
    server_free = [0.0]*c # χρόνος που είναι ελεύθερος κάθε εργαζόμενος
    heapq.heapify(server_free)

    t = 0.0
    waiting_times, system_times = [], []

    while True:
        t += random.expovariate(lam)
        if t > horizon_h:
            break

        free_time = heapq.heappop(server_free)
        start = max(t, free_time)
        w = start - t
        end = start + random.expovariate(mu)
        heapq.heappush(server_free, end)

        if t >= warmup_h:
            waiting_times.append(w)
            system_times.append(end - t)

    return {
        "mean_Wq_min": statistics.mean(waiting_times)*60,
        "mean_W_min": statistics.mean(system_times)*60,
        "p_wait": sum(1 for w in waiting_times if w > 0)/len(waiting_times),
        "p_W_lt_15": sum(1 for x in system_times if x < 15/60)/len(system_times)
    }
def simulate_mm_c_collect(lam, mu, c, horizon_h=60, warmup_h=10, seed=0):
    random.seed(seed)
    server_free = [0.0]*c
    heapq.heapify(server_free)

    t = 0.0
    arrival_times = []
    waiting_times = []
    system_times = []

    while True:
        t += random.expovariate(lam)
        if t > horizon_h:
            break

        free_time = heapq.heappop(server_free)
        start = max(t, free_time)
        w = start - t
        end = start + random.expovariate(mu)
        heapq.heappush(server_free, end)

        if t >= warmup_h:
            arrival_times.append(t)
            waiting_times.append(w)
            system_times.append(end - t)

    return arrival_times, waiting_times, system_times
def run_for_c_values(c_values):
    rows = []
    for c_ in c_values:
        m = mmc_metrics(lam, mu, c_)
        if not m["stable"]:
            rows.append((c_, m["Ro"], None, None, None, None, None))
            continue

        sla = sla_prob_mmc(lam, mu, c_, T15)
        cost = delay_cost_per_hour(lam, m["Wq_h"])
        rows.append((
            c_,
            m["Ro"],
            m["Pwait"],
            m["Wq_h"]*60,
            m["W_h"]*60,
            sla,
            cost
        ))
    return rows

print("\n=== ΘΕΩΡΗΤΙΚΑ ΑΠΟΤΕΛΕΣΜΑΤΑ (M/M/{}) ===".format(c))
m = mmc_metrics(lam, mu, c)

if not m["stable"]:
    print("Ro =", round(m["Ro"], 3))
    print("❌ Το σύστημα είναι ΑΣΤΑΘΕΣ (Ro ≥ 1)")
else:
    print("Ro =", round(m["Ro"], 3))
    print("P(wait) =", round(m["Pwait"], 4))
    print("Lq =", round(m["Lq"], 3))
    print("Wq =", round(m["Wq_h"]*60, 2), "λεπτά")
    print("W  =", round(m["W_h"]*60, 2), "λεπτά")

    sla = sla_prob_mmc(lam, mu, c, T15)
    print("SLA: P(W < 15') =", round(sla, 4))

    cost = delay_cost_per_hour(lam, m["Wq_h"])
    print("Κόστος καθυστέρησης €/ώρα =", round(cost, 2))

# ================== RUN SIMULATION ==================
print("\n=== ΠΡΟΣΟΜΟΙΩΣΗ (M/M/{}) ===".format(c))
sim = simulate_mm_c(lam, mu, c)
for k, v in sim.items():
    print(k, "=", round(v, 4))
# ================== ΣΥΓΚΡΙΣΗ ΠΟΛΛΩΝ c ==================

c_list = [3,4,5,6]
rows = run_for_c_values(c_list)

print("\nΣύγκριση M/M/c")
print("c | Ro | P(wait) | Wq(min) | W(min) | SLA P(W<15') | Cost €/h")
for r in rows:
    c_, Ro, Pwait, Wqmin, Wmin, sla, cost = r
    print(f"{c_:>1} | {Ro:>5.3f} | {('---' if Pwait is None else f'{Pwait:>6.3f}'):>6} | "
          f"{('---' if Wqmin is None else f'{Wqmin:>7.2f}'):>7} | "
          f"{('---' if Wmin is None else f'{Wmin:>6.2f}'):>6} | "
          f"{('---' if sla is None else f'{sla:>11.3f}'):>11} | "
          f"{('---' if cost is None else f'{cost:>8.2f}'):>8}")

# ================== G/G/1 APPROXIMATION  ==================
# Προσεγγιστικός τύπος από τις σημειώσεις (c = 1)

Ca = 1.1
Cs = 1.5

def gg1_notes_approx(lam, mu, Ca, Cs):
    Ro = lam / mu   # Για G/G/1: Ro = λ / μ

    if Ro >= 1:
        return {
            "stable": False,
            "Ro": Ro,
            "Wq_h": None,
            "W_h": None
        }

    # Από τις σημειώσεις:
    # p^d = Ro^{-1+sqrt(2(c+1))}, για c=1 => p^d = Ro
    pd = Ro

    V = (Ca**2 + Cs**2) / 2  # (Ca^2 + Cs^2)/2

    # W ≈ (1/μ) * V * (pd/(1-Ro)) + 1/μ
    W_h  = (1/mu) * V * (pd/(1 - Ro)) + (1/mu)
    Wq_h = W_h - (1/mu)

    return {
        "stable": True,
        "Ro": Ro,
        "Wq_h": Wq_h,
        "W_h": W_h
    }

gg1 = gg1_notes_approx(lam, mu, Ca, Cs)

print("\n=== G/G/1 APPROXIMATION (από σημειώσεις) ===")
print("Ro =", round(gg1["Ro"], 3))

if not gg1["stable"]:
    print("❌ ΑΣΤΑΘΕΣ: Ro ≥ 1")
    print("➡️ Η ουρά αυξάνεται χωρίς όριο, Wq και W δεν ορίζονται.")
else:
    print("Wq =", round(gg1["Wq_h"]*60, 2), "λεπτά")
    print("W  =", round(gg1["W_h"]*60, 2), "λεπτά")




